In [7]:
# %pip install torchaudio torch numpy chromadb
# %pip install ffmpeg-python


In [8]:
import os
import torch   ## library to handle tensors and audio processing
import torchaudio  ## library for audio processing
import torchaudio.transforms as T  ## library for audio transformations
import numpy as np
import chromadb
from chromadb.config import Settings
import subprocess
import os
import librosa   ### library to logging.info audio properties
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler()]  # Stream to stdout (Jupyter output)
)

logging.disable(logging.CRITICAL)
# torchaudio.set_audio_backend("ffmpeg")

# Set the backend to 'ffmpeg'
# torchaudio.set_audio_backend("ffmpeg")

# Or to 'sox'
# torchaudio.set_audio_backend("sox")

# Or to 'soundfile'
# torchaudio.set_audio_backend("soundfile")


# Settings
AUDIO_DIR = r"C:\genai\Audio"
SAMPLE_RATE = 16000
COLLECTION_NAME = r"audio_embeddings"

def analyze_audio(file_path):
    logging.info(f"\n🎵 Analyzing: {file_path}")

    # Load audio
    waveform, sample_rate = torchaudio.load(file_path)
    waveform = waveform.mean(dim=0).numpy()  # Convert to mono

    duration = len(waveform) / sample_rate
    logging.info(f"⏱️ Duration: {duration:.2f} seconds")
    logging.info(f"🎚️ Sample Rate: {sample_rate} Hz")

    # Compute RMS (Root Mean Square)
    rms = librosa.feature.rms(y=waveform)[0]
    avg_rms = np.mean(rms)
    logging.info(f"📶 RMS Energy (Intensity): {avg_rms:.6f}")

    # Convert RMS to decibels
    db = librosa.amplitude_to_db(rms, ref=np.max)
    avg_db = np.mean(db)
    logging.info(f"🔊 Average Loudness (dB): {avg_db:.2f} dB")

    # Estimate pitch (fundamental frequency)
    pitches, magnitudes = librosa.piptrack(y=waveform, sr=sample_rate)
    pitch_values = pitches[magnitudes > np.median(magnitudes)]
    pitch_values = pitch_values[pitch_values > 0]

    if len(pitch_values) > 0:
        avg_pitch = np.mean(pitch_values)
        logging.info(f"🎵 Estimated Average Pitch: {avg_pitch:.2f} Hz")
    else:
        logging.info("🎵 Estimated Pitch: Not Detected")

def convert_to_wav(input_path):

    if not input_path.lower().endswith(".mp3"):
        return input_path  # Skip if already wav

    logging.info("✅ Python Code to logging.info Audio Properties")
    analyze_audio(input_path)
    output_path = input_path.rsplit(".", 1)[0] + ".wav"
    if not os.path.exists(output_path):
        try:
            subprocess.run(["ffmpeg", "-y", "-i", input_path, output_path],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except FileNotFoundError:
            raise RuntimeError("FFmpeg not found. Please install and add to PATH.")
    return output_path


# Extract MFCC features from audio file
def extract_features(filepath, sample_rate=SAMPLE_RATE):
    logging.info(f"📂 func extract_features **->  Processing: {filepath}")
    # waveform, sr = torchaudio.load(filepath)

    # filepath = convert_to_wav(filepath)
    for fname in os.listdir(AUDIO_DIR):
        fpath = os.path.join(AUDIO_DIR, fname)    
        if not fpath.lower().endswith(('.mp3')):
            waveform, sr = torchaudio.load(fpath)    

            logging.info(f"Sample Rate: {sr}, Shape: {waveform.shape}")
            if sr != sample_rate:
                resampler = T.Resample(sr, sample_rate)
                waveform = resampler(waveform)

            logging.info(f"Resampled Shape: {waveform.shape}")
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)

            logging.info(f"Mono Shape: {waveform.shape}")
            mfcc_transform = T.MFCC(
                sample_rate=sample_rate,
                n_mfcc=80,
                melkwargs={"n_fft": 400, "hop_length": 160, "n_mels": 100, "center": False}
            )

            mfcc = mfcc_transform(waveform)
            return mfcc.mean(dim=2).squeeze().numpy().tolist()

# Initialize ChromaDB collection
def init_chroma_collection():
    client = chromadb.Client(Settings(anonymized_telemetry=False))
    if COLLECTION_NAME not in [col.name for col in client.list_collections()]:
        collection = client.create_collection(name=COLLECTION_NAME)
    else:
        collection = client.get_collection(name=COLLECTION_NAME)
    return collection

# Index stored audio files into ChromaDB
def index_audio_files(collection):
    logging.info("🔄 Indexing stored audio files...")

    for fname in os.listdir(AUDIO_DIR):
        fpath = os.path.join(AUDIO_DIR, fname)
        if not fpath.lower().endswith(( '.mp3', '.flac')):
            continue

        doc_id = fname
        existing = collection.get(ids=[doc_id], include=["embeddings"])
        if existing["ids"]:  # Already indexed
            continue

        try:
            emb = extract_features(fpath)
            collection.add(documents=[fname], embeddings=[emb], ids=[doc_id])
            logging.info(f"✅ Indexed: {fname}")
        except Exception as e:
            logging.info(f"❌ Failed {fname}: {e}")

# Find top-k similar audios
def find_similar_audio(uploaded_file, collection, k=5):
    logging.info(f"🔍 func find_similar_audio **> before query_embFinding**-> similar audios for: {uploaded_file}")
    query_emb = extract_features(uploaded_file)
    if not query_emb:
        logging.info("❌ Failed to extract features from the uploaded file.")
        return
    results = collection.query(query_embeddings=[query_emb], n_results=k)

    logging.info("\n🔍 Top Matches:")
    for doc, dist in zip(results['documents'][0], results['distances'][0]):
        print(f"{doc}: score = {1 - dist:.4f}")

# Entry point
if __name__ == "__main__":
    uploaded_file = AUDIO_DIR   #input("📁 Enter path to the uploaded audio file: ").strip()
    if not os.path.exists(uploaded_file):
        logging.info("❌ File not found.")
        exit()

    collection = init_chroma_collection()
    index_audio_files(collection)
    logging.info("🔄 Indexing complete. Now finding similar audios...")
    find_similar_audio(uploaded_file, collection)


astral-creepy-dark-logo-254198.mp3: score = -4772.0942
comedy-music-theme-121124.mp3: score = -4772.0942
dont-talk-315229.mp3: score = -4772.0942
future-design-344320.mp3: score = -4772.0942
indian-bansuri-tabla-fusion-short.mp3: score = -4772.0942


c:\genai\gen3\Lib\site-packages\torchaudio\functional\functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (100) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(


In [9]:
from sklearn.metrics.pairwise import cosine_similarity
import os
import torch   ## library to handle tensors and audio processing
import torchaudio  ## library for audio processing
import torchaudio.transforms as T  ## library for audio transformations
import numpy as np
import chromadb
from chromadb.config import Settings
import subprocess
import os
import librosa   ### library to logging.info audio properties
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler()]  # Stream to stdout (Jupyter output)
)

logging.disable(logging.CRITICAL)
# torchaudio.set_audio_backend("ffmpeg")

# Set the backend to 'ffmpeg'
# torchaudio.set_audio_backend("ffmpeg")

# Or to 'sox'
# torchaudio.set_audio_backend("sox")

# Or to 'soundfile'
# torchaudio.set_audio_backend("soundfile")


# Settings
AUDIO_DIR = r"C:\genai\Audio"
SAMPLE_RATE = 16000
COLLECTION_NAME = r"audio_embeddings"

def analyze_audio(file_path):
    logging.info(f"\n🎵 Analyzing: {file_path}")

    # Load audio
    waveform, sample_rate = torchaudio.load(file_path)
    waveform = waveform.mean(dim=0).numpy()  # Convert to mono

    duration = len(waveform) / sample_rate
    logging.info(f"⏱️ Duration: {duration:.2f} seconds")
    logging.info(f"🎚️ Sample Rate: {sample_rate} Hz")

    # Compute RMS (Root Mean Square)
    rms = librosa.feature.rms(y=waveform)[0]
    avg_rms = np.mean(rms)
    logging.info(f"📶 RMS Energy (Intensity): {avg_rms:.6f}")

    # Convert RMS to decibels
    db = librosa.amplitude_to_db(rms, ref=np.max)
    avg_db = np.mean(db)
    logging.info(f"🔊 Average Loudness (dB): {avg_db:.2f} dB")

    # Estimate pitch (fundamental frequency)
    pitches, magnitudes = librosa.piptrack(y=waveform, sr=sample_rate)
    pitch_values = pitches[magnitudes > np.median(magnitudes)]
    pitch_values = pitch_values[pitch_values > 0]

    if len(pitch_values) > 0:
        avg_pitch = np.mean(pitch_values)
        logging.info(f"🎵 Estimated Average Pitch: {avg_pitch:.2f} Hz")
    else:
        logging.info("🎵 Estimated Pitch: Not Detected")

def convert_to_wav(input_path):

    if not input_path.lower().endswith(".mp3"):
        return input_path  # Skip if already wav

    logging.info("✅ Python Code to logging.info Audio Properties")
    analyze_audio(input_path)
    output_path = input_path.rsplit(".", 1)[0] + ".wav"
    if not os.path.exists(output_path):
        try:
            subprocess.run(["ffmpeg", "-y", "-i", input_path, output_path],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except FileNotFoundError:
            raise RuntimeError("FFmpeg not found. Please install and add to PATH.")
    return output_path


# Extract MFCC features from audio file
def extract_features(filepath, sample_rate=SAMPLE_RATE):
    logging.info(f"📂 func extract_features **->  Processing: {filepath}")
    # waveform, sr = torchaudio.load(filepath)

    # filepath = convert_to_wav(filepath)
    for fname in os.listdir(AUDIO_DIR):
        fpath = os.path.join(AUDIO_DIR, fname)    
        if not fpath.lower().endswith(('.mp3')):
            waveform, sr = torchaudio.load(fpath)    

            logging.info(f"Sample Rate: {sr}, Shape: {waveform.shape}")
            if sr != sample_rate:
                resampler = T.Resample(sr, sample_rate)
                waveform = resampler(waveform)

            logging.info(f"Resampled Shape: {waveform.shape}")
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)

            logging.info(f"Mono Shape: {waveform.shape}")
            mfcc_transform = T.MFCC(
                sample_rate=sample_rate,
                n_mfcc=80,
                melkwargs={"n_fft": 400, "hop_length": 160, "n_mels": 100, "center": False}
            )

            mfcc = mfcc_transform(waveform)
            return mfcc.mean(dim=2).squeeze().numpy().tolist()

# Initialize ChromaDB collection
def init_chroma_collection():
    client = chromadb.Client(Settings(anonymized_telemetry=False))
    if COLLECTION_NAME not in [col.name for col in client.list_collections()]:
        collection = client.create_collection(name=COLLECTION_NAME)
    else:
        collection = client.get_collection(name=COLLECTION_NAME)
    return collection

# Index stored audio files into ChromaDB
def index_audio_files(collection):
    logging.info("🔄 Indexing stored audio files...")

    for fname in os.listdir(AUDIO_DIR):
        fpath = os.path.join(AUDIO_DIR, fname)
        if not fpath.lower().endswith(( '.mp3', '.flac')):
            continue

        doc_id = fname
        existing = collection.get(ids=[doc_id], include=["embeddings"])
        if existing["ids"]:  # Already indexed
            continue

        try:
            emb = extract_features(fpath)
            collection.add(documents=[fname], embeddings=[emb], ids=[doc_id])
            logging.info(f"✅ Indexed: {fname}")
        except Exception as e:
            logging.info(f"❌ Failed {fname}: {e}")

# Find top-k similar audios
def find_similar_audio(uploaded_file, collection, k=5):
    logging.info(f"🔍 func find_similar_audio **> before query_embFinding**-> similar audios for: {uploaded_file}")
    query_emb = extract_features(uploaded_file)
    if not query_emb:
        logging.info("❌ Failed to extract features from the uploaded file.")
        return
    results = collection.query(query_embeddings=[query_emb], n_results=k)
    print(results)
    logging.info("\n🔍 Top Matches:")
    for doc, dist in zip(results['documents'][0], results['distances'][0]):
        print(f"{doc}: score = {1 - dist:.4f}")

# Entry point
if __name__ == "__main__":
    uploaded_file = AUDIO_DIR   #input("📁 Enter path to the uploaded audio file: ").strip()
    if not os.path.exists(uploaded_file):
        logging.info("❌ File not found.")
        exit()

    collection = init_chroma_collection()
    index_audio_files(collection)
    logging.info("🔄 Indexing complete. Now finding similar audios...")
    find_similar_audio(uploaded_file, collection)


{'ids': [['astral-creepy-dark-logo-254198.mp3', 'comedy-music-theme-121124.mp3', 'dont-talk-315229.mp3', 'future-design-344320.mp3', 'indian-bansuri-tabla-fusion-short.mp3']], 'embeddings': None, 'documents': [['astral-creepy-dark-logo-254198.mp3', 'comedy-music-theme-121124.mp3', 'dont-talk-315229.mp3', 'future-design-344320.mp3', 'indian-bansuri-tabla-fusion-short.mp3']], 'uris': None, 'data': None, 'metadatas': [[None, None, None, None, None]], 'distances': [[4773.09423828125, 4773.09423828125, 4773.09423828125, 4773.09423828125, 4773.09423828125]], 'included': [<IncludeEnum.distances: 'distances'>, <IncludeEnum.documents: 'documents'>, <IncludeEnum.metadatas: 'metadatas'>]}
astral-creepy-dark-logo-254198.mp3: score = -4772.0942
comedy-music-theme-121124.mp3: score = -4772.0942
dont-talk-315229.mp3: score = -4772.0942
future-design-344320.mp3: score = -4772.0942
indian-bansuri-tabla-fusion-short.mp3: score = -4772.0942


c:\genai\gen3\Lib\site-packages\torchaudio\functional\functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (100) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(
